# 2. Prepare the aligned analysis dataset — corrected

All sensor tables are aligned to the native 10-s acquisition grid before the wide dataset is created. This avoids matching sensors on exact timestamps when their clocks differ by a small fraction of the acquisition interval.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Project root not found. Run the notebook from inside the project repository.")

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def report_dir(root: Path, analysis_name: str, subfolder: str | None = None) -> Path:
    base = root / "reports" / analysis_name
    return ensure_dir(base if subfolder is None else base / subfolder)

ROOT = find_project_root()
print("ROOT:", ROOT)


ROOT: /home/eleonorab/repository/convective_cooling


In [2]:
# =====================
# CONFIGURATION
# =====================

INPUT_DIR = ROOT / "data" / "processed" / "aggregated_by_signal"
OUTPUT_DIR = ROOT / "data" / "processed" / "analysis"
TIME_STEP_SECONDS = 10

ensure_dir(OUTPUT_DIR)

PHASE_BOUNDS = {
    "standing_1": (0, 600),
    "walk_low_1": (600, 1200),
    "walk_mod_1": (1200, 1800),
    "walk_low_2": (1800, 2400),
    "walk_mod_2": (2400, 3000),
    "standing_2": (3000, 3600),
}


def assign_phase(elapsed_seconds: float) -> str | float:
    for phase, (start, end) in PHASE_BOUNDS.items():
        if start <= elapsed_seconds < end:
            return phase
    return np.nan


# =====================
# LOAD SIGNAL TABLES
# =====================

signal_tables = []

excluded_input_files = {
    "dataset_build_summary.csv",
}

for file in sorted(INPUT_DIR.glob("*.csv")):
    if file.name in excluded_input_files:
        print(f"Skipping metadata file: {file.name}")
        continue

    signal = file.stem.lower().strip()
    table = pd.read_csv(file)

    required = {
        "subject_id", "condition", "timestamp",
        "elapsed_seconds", "elapsed_minutes", "value",
    }
    missing = required.difference(table.columns)
    if missing:
        raise ValueError(f"Missing columns in {file}: {sorted(missing)}")

    table = table[list(required)].copy()
    table["subject_id"] = table["subject_id"].astype(str).str.strip()
    table["condition"] = table["condition"].astype(str).str.strip().str.lower()
    table["elapsed_seconds"] = pd.to_numeric(table["elapsed_seconds"], errors="coerce")
    table["value"] = pd.to_numeric(table["value"], errors="coerce")
    table["timestamp"] = pd.to_datetime(table["timestamp"], errors="coerce")
    table = table.dropna(subset=["subject_id", "condition", "elapsed_seconds", "value"])
    table = table.loc[table["elapsed_seconds"].between(0, 3599)].copy()

    # Native acquisition frequency is one observation every 10 s.
    # Rounding only aligns small clock offsets between sensors.
    table["elapsed_seconds_bin"] = (
        np.round(table["elapsed_seconds"] / TIME_STEP_SECONDS)
        * TIME_STEP_SECONDS
    ).astype(int)
    table = table.loc[table["elapsed_seconds_bin"].between(0, 3590)].copy()
    table["signal"] = signal
    signal_tables.append(table)

if not signal_tables:
    raise FileNotFoundError(f"No signal CSV files found in {INPUT_DIR}")

df_long_raw = pd.concat(signal_tables, ignore_index=True)


# =====================
# ONE VALUE PER SIGNAL AND 10-s BIN
# =====================

df_long = (
    df_long_raw
    .groupby(
        ["subject_id", "condition", "signal", "elapsed_seconds_bin"],
        as_index=False,
        observed=True,
    )
    .agg(
        value=("value", "mean"),
        timestamp=("timestamp", "min"),
    )
)

df_long["elapsed_seconds"] = df_long["elapsed_seconds_bin"].astype(float)
df_long["elapsed_minutes"] = df_long["elapsed_seconds"] / 60.0
df_long["phase"] = df_long["elapsed_seconds"].apply(assign_phase)
df_long["time_bin_5min"] = np.floor(df_long["elapsed_minutes"] / 5.0) * 5.0
df_long = df_long.dropna(subset=["phase"]).copy()

df_long.to_csv(OUTPUT_DIR / "all_signals_long.csv", index=False)


# =====================
# WIDE DATASET
# =====================

df_wide = (
    df_long
    .pivot_table(
        index=["subject_id", "condition", "elapsed_seconds_bin"],
        columns="signal",
        values="value",
        aggfunc="mean",
        observed=True,
    )
    .reset_index()
)

df_wide["elapsed_seconds"] = df_wide["elapsed_seconds_bin"].astype(float)
df_wide["elapsed_minutes"] = df_wide["elapsed_seconds"] / 60.0
df_wide["phase"] = df_wide["elapsed_seconds"].apply(assign_phase)
df_wide["time_bin_5min"] = np.floor(df_wide["elapsed_minutes"] / 5.0) * 5.0

if "t_microclimateback" in df_wide and "t_microclimatefront" in df_wide:
    df_wide["t_microclimate_gradient"] = (
        df_wide["t_microclimateback"] - df_wide["t_microclimatefront"]
    )

if "rh_microclimateback" in df_wide and "rh_microclimatefront" in df_wide:
    df_wide["rh_microclimate_gradient"] = (
        df_wide["rh_microclimateback"] - df_wide["rh_microclimatefront"]
    )

df_wide = df_wide.sort_values(
    ["subject_id", "condition", "elapsed_seconds"]
).reset_index(drop=True)

df_wide.to_csv(OUTPUT_DIR / "analysis_dataset_wide.csv", index=False)

print("Analysis dataset created successfully")
print("Long shape:", df_long.shape)
print("Wide shape:", df_wide.shape)
print("Signals:", sorted(df_long["signal"].unique()))
print("Rows per subject-condition:")
print(df_wide.groupby(["subject_id", "condition"], observed=True).size().describe())


Skipping metadata file: dataset_build_summary.csv


/tmp/ipykernel_32574/1377895299.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_long_raw = pd.concat(signal_tables, ignore_index=True)


Analysis dataset created successfully
Long shape: (203710, 10)
Wide shape: (10800, 29)
Signals: ['rh_abdomen', 'rh_chest', 'rh_dorsal', 'rh_environment', 'rh_hand', 'rh_microclimateback', 'rh_microclimatefront', 'rh_neck', 'rh_scapula', 'rh_shin', 't_abdomen', 't_chest', 't_dorsal', 't_environment', 't_hand', 't_microclimateback', 't_microclimatefront', 't_neck', 't_scapula', 't_shin']
Rows per subject-condition:
count     30.0
mean     360.0
std        0.0
min      360.0
25%      360.0
50%      360.0
75%      360.0
max      360.0
dtype: float64
